In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("E-Commerce").getOrCreate()

In [3]:
from google.colab import files
uploaded = files.upload()

Saving logs_txt.txt to logs_txt.txt
Saving order_items.csv to order_items.csv
Saving payments.csv to payments.csv
Saving orders.csv to orders.csv
Saving products.csv to products.csv
Saving customers.csv to customers.csv
Saving sales.csv to sales.csv


In [4]:
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)
orders = spark.read.csv("orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv("order_items.csv", header=True, inferSchema=True)
products = spark.read.csv("products.csv", header=True, inferSchema=True)
payments = spark.read.csv("payments.csv", header=True, inferSchema=True)

In [5]:
full_df = customers.join(orders, "customer_id") \
                   .join(order_items, "order_id") \
                   .join(products, "product_id") \
                   .join(payments, "order_id")

In [6]:
from pyspark.sql.functions import col

full_df = full_df.withColumn("revenue", col("quantity") * col("price"))
full_df.show()

+--------+----------+-----------+------+---------+---+-----------+----------+---------+--------+------------+-----------+-----+----------+------------+------+-------+
|order_id|product_id|customer_id|  name|     city|age|signup_date|order_date|   status|quantity|product_name|   category|price|payment_id|payment_type|amount|revenue|
+--------+----------+-----------+------+---------+---+-----------+----------+---------+--------+------------+-----------+-----+----------+------------+------+-------+
|       1|       102|          1|  Amit|Hyderabad| 28| 2023-01-10|2024-03-01|Delivered|       2|  Headphones|Electronics| 3000|         1| Credit Card| 81000|   6000|
|       1|       101|          1|  Amit|Hyderabad| 28| 2023-01-10|2024-03-01|Delivered|       1|      Laptop|Electronics|75000|         1| Credit Card| 81000|  75000|
|       2|       103|          2| Priya|Bangalore| 32| 2023-02-12|2024-03-02|Delivered|       1|    Keyboard|Electronics| 1500|         2|         UPI|  1500|   1500

In [7]:
from pyspark.sql.functions import sum

top_customers = full_df.groupBy("name") \
                       .sum("revenue") \
                       .withColumnRenamed("sum(revenue)", "total_revenue") \
                       .orderBy(col("total_revenue").desc())

top_customers.show()

+------+-------------+
|  name|total_revenue|
+------+-------------+
|  Amit|        81000|
| Rahul|        75000|
| Arjun|        40000|
|  Neha|        30000|
| Sneha|        12000|
| Meera|         9000|
| Karan|         3000|
| Priya|         1500|
| Divya|          500|
|Vikram|          200|
+------+-------------+



In [8]:
top_products = full_df.groupBy("product_name") \
                      .sum("revenue") \
                      .withColumnRenamed("sum(revenue)", "total_revenue") \
                      .orderBy(col("total_revenue").desc())

top_products.show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|      Laptop|       150000|
|  Smartphone|        40000|
|      Tablet|        30000|
|  Headphones|        15000|
|     Monitor|        12000|
|    Keyboard|         4500|
|    Notebook|          500|
|         Pen|          200|
+------------+-------------+



In [9]:
city_revenue = full_df.groupBy("city") \
                      .sum("revenue") \
                      .withColumnRenamed("sum(revenue)", "total_revenue") \
                      .orderBy(col("total_revenue").desc())

city_revenue.show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Hyderabad|        90000|
|   Mumbai|        75200|
|    Delhi|        42000|
|  Chennai|        40000|
|     Pune|         3000|
|Bangalore|         2000|
+---------+-------------+



In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window_spec = Window.orderBy(col("total_revenue").desc())

ranked_customers = top_customers.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

ranked_customers.show()

+------+-------------+----+
|  name|total_revenue|rank|
+------+-------------+----+
|  Amit|        81000|   1|
| Rahul|        75000|   2|
| Arjun|        40000|   3|
|  Neha|        30000|   4|
| Sneha|        12000|   5|
| Meera|         9000|   6|
| Karan|         3000|   7|
| Priya|         1500|   8|
| Divya|          500|   9|
|Vikram|          200|  10|
+------+-------------+----+



In [11]:
final_df = ranked_customers
final_df.show()

+------+-------------+----+
|  name|total_revenue|rank|
+------+-------------+----+
|  Amit|        81000|   1|
| Rahul|        75000|   2|
| Arjun|        40000|   3|
|  Neha|        30000|   4|
| Sneha|        12000|   5|
| Meera|         9000|   6|
| Karan|         3000|   7|
| Priya|         1500|   8|
| Divya|          500|   9|
|Vikram|          200|  10|
+------+-------------+----+



In [12]:
final_df.write.mode("overwrite").csv("ecommerce_report", header=True)